# Notebook 09: Differential Drive Kinematics (Motion Model)

This notebook explains the robot motion model used in `kiss_slam`, from pose representation to numerical integration details.

> Code connection: `src/kiss_slam/models/motion.py` (`DifferentialDriveMotionModel`, `unicycle_predict_state`, `unicycle_jacobians`).

## Table of Contents

- [Learning objectives](#learning-objectives)
- [Prerequisites](#prerequisites)
- [1) Pose $(x, y, \theta)$](#1-pose-x-y-theta)
- [2) Control inputs and one-step integration](#2-control-inputs-and-one-step-integration)
- [3) Straight motion vs arc motion](#3-straight-motion-vs-arc-motion)
- [4) Why wrapping angles matters](#4-why-wrapping-angles-matters)
- [5) Interactive trajectory simulator](#5-interactive-trajectory-simulator)
- [6) Sensitivity to timestep `dt`](#6-sensitivity-to-timestep-dt)
- [7) Exercise: implement your own `integrate_step`](#7-exercise-implement-your-own-integratestep)
  - [Task](#task)
  - [Optional solution hint](#optional-solution-hint)
- [Recap](#recap)

---

## Navigation

- Previous: [ntbk-08-ekf-consistency-nis-nees.ipynb](./ntbk-08-ekf-consistency-nis-nees.ipynb)
- Next: [ntbk-10-motion-model-jacobians-and-process-noise.ipynb](./ntbk-10-motion-model-jacobians-and-process-noise.ipynb)

## Relevant source files (repo paths)
- `src/kiss_slam/models/motion.py`


In [ ]:
# Notebook setup (reproducible + matplotlib defaults)
import numpy as np
import matplotlib.pyplot as plt
from _notebook_utils import set_seed

SEED = 108
set_seed(SEED)
plt.rcdefaults()


## Learning objectives

By the end, you should be able to:

1. Explain robot pose as $(x, y, \theta)$.
2. Use control inputs $(v, \omega)$ to integrate one motion step.
3. Distinguish straight vs arc motion.
4. Understand why angle wrapping is necessary.
5. Compare your own `integrate_step` with the repo motion model.
6. See how timestep `dt` affects trajectory accuracy.

## Prerequisites

- Basic Python + NumPy.
- Trigonometry (`sin`, `cos`, `atan2`).
- Very basic idea of an EKF predict step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from kiss_slam.models.motion import DifferentialDriveMotionModel
from kiss_slam.types import ControlInput
from kiss_slam.math_utils import wrap_angle

SEED = 9
np.random.seed(SEED)
print(f"Seed set to {SEED}")


## 1) Pose $(x, y, \theta)$

A 2D robot pose contains:
- `x`: horizontal world position,
- `y`: vertical world position,
- `theta`: heading angle (radians).

In this repo, the motion model state is a NumPy vector of length 3.

In [ ]:
pose = np.array([1.0, 2.0, np.deg2rad(30.0)], dtype=float)
print("pose =", pose)
print("x, y, theta(deg) =", pose[0], pose[1], np.rad2deg(pose[2]))


## 2) Control inputs and one-step integration

For differential-drive / unicycle kinematics used here:

$$
\begin{aligned}
x_{k+1} &= x_k + v\,dt\cos\theta_k \\ny_{k+1} &= y_k + v\,dt\sin\theta_k \\n\theta_{k+1} &= \operatorname{wrap}(\theta_k + \omega\,dt)
\end{aligned}
$$

where:
- `v` is linear velocity (m/s),
- `omega` is yaw rate (rad/s),
- `dt` is timestep.

In [ ]:
model = DifferentialDriveMotionModel()

pose0 = np.array([0.0, 0.0, 0.0], dtype=float)
u = ControlInput(v=1.0, w=0.4)
dt = 0.1

pose1 = model.predict_state(pose0, u, dt)
print("pose0:", pose0)
print("pose1:", pose1)


## 3) Straight motion vs arc motion

- If $\omega = 0$: heading is constant, so robot moves in a straight line.
- If $\omega \neq 0$: heading changes each step, so robot follows an arc.

Let's simulate both.

In [ ]:
def rollout(model, pose0, control, dt, steps):
    poses = [pose0.copy()]
    pose = pose0.copy()
    for _ in range(steps):
        pose = model.predict_state(pose, control, dt)
        poses.append(pose.copy())
    return np.asarray(poses)

pose0 = np.array([0.0, 0.0, 0.0], dtype=float)
steps = 120

t_straight = rollout(model, pose0, ControlInput(v=1.0, w=0.0), dt=0.1, steps=steps)
t_arc = rollout(model, pose0, ControlInput(v=1.0, w=0.5), dt=0.1, steps=steps)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(t_straight[:, 0], t_straight[:, 1], label="straight (w=0)")
ax.plot(t_arc[:, 0], t_arc[:, 1], label="arc (w=0.5)")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("Differential-drive trajectories")
ax.grid(True)
ax.legend();


## 4) Why wrapping angles matters

Without wrapping, heading can grow beyond $\pi$ or below $-\pi$, making angle differences ambiguous.

This repo wraps headings into $[-\pi, \pi)$ with `kiss_slam.math_utils.wrap_angle`.

In [ ]:
angles = np.array([
    -4*np.pi,
    -3.2,
    -np.pi,
    -0.1,
    0.0,
    np.pi - 1e-6,
    np.pi,
    3.7,
    5*np.pi,
])
wrapped = wrap_angle(angles)

for a, w in zip(angles, wrapped):
    print(f"raw={a: .3f} -> wrapped={w: .3f}")


## 5) Interactive trajectory simulator

Try different controls and timestep values to build intuition.

- Increase `v` to move faster.
- Increase `w` to curve harder.
- Decrease `dt` to get finer integration.

In [ ]:
# --- interactive knobs (edit and re-run) ---
v = 1.0          # linear speed [m/s]
w = 0.7          # yaw rate [rad/s]
dt = 0.2         # timestep [s]
steps = 80       # number of integration steps
theta0_deg = 20  # initial heading [deg]

pose0 = np.array([0.0, 0.0, np.deg2rad(theta0_deg)], dtype=float)
traj = rollout(model, pose0, ControlInput(v=v, w=w), dt=dt, steps=steps)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(traj[:, 0], traj[:, 1], marker=".", markersize=3)
axes[0].scatter([traj[0,0]], [traj[0,1]], c="green", label="start")
axes[0].scatter([traj[-1,0]], [traj[-1,1]], c="red", label="end")
axes[0].set_aspect("equal", adjustable="box")
axes[0].set_xlabel("x [m]")
axes[0].set_ylabel("y [m]")
axes[0].set_title("Trajectory in world frame")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(np.rad2deg(traj[:, 2]))
axes[1].set_xlabel("step")
axes[1].set_ylabel("theta [deg]")
axes[1].set_title("Heading evolution (wrapped)")
axes[1].grid(True)

plt.tight_layout()
print("final pose:", traj[-1])


## 6) Sensitivity to timestep `dt`

For the same total simulated time, changing `dt` changes discrete integration behavior.

The model in `motion.py` is first-order Euler integration, so larger `dt` can add noticeable discretization error.

In [ ]:
pose0 = np.array([0.0, 0.0, 0.0], dtype=float)
control = ControlInput(v=1.0, w=1.0)
T = 8.0

dt_list = [0.5, 0.2, 0.1, 0.05]

fig, ax = plt.subplots(figsize=(6, 5))
for dt in dt_list:
    steps = int(T / dt)
    tr = rollout(model, pose0, control, dt=dt, steps=steps)
    ax.plot(tr[:,0], tr[:,1], label=f"dt={dt}")

ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("Trajectory sensitivity to dt")
ax.grid(True)
ax.legend();


## 7) Exercise: implement your own `integrate_step`

Try implementing the same equations yourself, then compare against repo outputs.

### Task
1. Complete `integrate_step_student`.
2. Run `compare_integrators(...)`.
3. Confirm max absolute difference is near numerical precision.

In [ ]:
def integrate_step_student(pose, v, w, dt):
    """Student implementation of one unicycle step."""
    x, y, theta = pose
    x_next = x + v * dt * np.cos(theta)
    y_next = y + v * dt * np.sin(theta)
    theta_next = wrap_angle(theta + w * dt)
    return np.array([x_next, y_next, theta_next], dtype=float)


def compare_integrators(n_trials=50):
    max_err = 0.0
    for _ in range(n_trials):
        pose = np.array([
            np.random.uniform(-5, 5),
            np.random.uniform(-5, 5),
            np.random.uniform(-np.pi, np.pi),
        ])
        v = np.random.uniform(-2.0, 2.0)
        w = np.random.uniform(-2.5, 2.5)
        dt = np.random.uniform(0.01, 0.3)

        ours = integrate_step_student(pose, v, w, dt)
        repo = model.predict_state(pose, ControlInput(v=v, w=w), dt)
        err = np.max(np.abs(ours - repo))
        max_err = max(max_err, float(err))
    return max_err

max_err = compare_integrators(n_trials=200)
print(f"max |ours - repo| = {max_err:.3e}")


### Optional solution hint

If your implementation differs, inspect:
- sign of `sin/cos` terms,
- whether heading uses old or new angle,
- whether angle wrapping is applied.

## Recap

You learned how this repo's differential-drive motion model works in practice:
- pose representation $(x,y,\theta)$,
- controls $(v,\omega)$ and one-step integration,
- straight vs arc trajectories,
- angle wrapping,
- timestep sensitivity,
- and how to validate your own implementation against `DifferentialDriveMotionModel`.

## Exercise Solutions

<details>
<summary>Show solution sketches</summary>

- Re-run the exercise code cells and compare your intermediate values to the reference outputs printed in this notebook.
- Check shapes (`mean`, `covariance`, Jacobians) first; most EKF mistakes are shape/sign issues.
- Use the same `SEED` from the setup cell to keep your run deterministic while debugging.

</details>
